In [13]:
# Import the neccessary Libraries
import pandas as pd

import re
import pandas as pd
from Bio import SeqIO

Parsing the UC File

In [3]:
uc_cols = ["type","cluster","qlen","pct_id","strand","x1","x2","cigar","query","target"]
uc = pd.read_csv("closed_ref_map.uc", sep="\t", header=None, names=uc_cols)

# Keep only hit lines
hits = uc[uc["type"] == "H"][["query", "target", "pct_id"]].copy()
hits.columns = ["asv_id", "gtdb_ref", "pct_identity"]

print(f"Total ASVs with hits: {len(hits)}")
hits.head()

Total ASVs with hits: 68341


,asv_id,gtdb_ref,pct_identity
0,PRJNA266806_ASV1,RS_GCF_002274615.1~NZ_LLAG01000108.1,100.0
1,PRJNA266806_ASV8,RS_GCF_025187565.1~NZ_RIJA01000015.1,100.0
2,PRJNA266806_ASV9,RS_GCF_934855955.1~NZ_CAKWVU010000076.1,100.0
3,PRJNA266806_ASV2,RS_GCF_959025655.1~NZ_CAUFJN010000109.1,100.0
4,PRJNA266806_ASV5,RS_GCF_002274615.1~NZ_LLAG01000108.1,100.0


Checking for unmatched ASVs

In [7]:
lookup_df = pd.read_csv("asv_lookup_master.csv")
lookup_df.head()

,asv_id,sequence,study
0,PRJNA266806_ASV1,GATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAACGCTTTG...,PRJNA266806
1,PRJNA266806_ASV2,GATGAACGCTGGCGGCGTGCCTAACACATGCAAGTCGAGCGATTCT...,PRJNA266806
2,PRJNA266806_ASV3,GATGAACGCTAGCTACAGGCTTAACACATGCAAGTCGAGGGGCAGC...,PRJNA266806
3,PRJNA266806_ASV4,GATGAACGCTAGCTACAGGCTTAACACATGCAAGTCGAGGGGCAGC...,PRJNA266806
4,PRJNA266806_ASV5,GATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAACCA...,PRJNA266806


In [9]:
print(f"Total ASVs searched: {len(lookup_df)}")
print(f"ASVs matched: {len(hits)}")
print(f"ASVs unmatched: {len(lookup_df) - len(hits)}")

Total ASVs searched: 317561
ASVs matched: 68341
ASVs unmatched: 249220


In [16]:
print(hits["gtdb_ref"].head(10))
print(hits["gtdb_ref"].iloc[0])
print(repr(hits["gtdb_ref"].iloc[0]))  # repr shows hidden characters/whitespace

0       RS_GCF_002274615.1~NZ_LLAG01000108.1
1       RS_GCF_025187565.1~NZ_RIJA01000015.1
2    RS_GCF_934855955.1~NZ_CAKWVU010000076.1
3    RS_GCF_959025655.1~NZ_CAUFJN010000109.1
4       RS_GCF_002274615.1~NZ_LLAG01000108.1
5       GB_GCA_023447495.1~JAEXNC010000228.1
6       GB_GCA_022781305.1~JALETV010000081.1
7       GB_GCA_949083875.1~CARVQC010000008.1
8          GB_GCA_003202955.1~BFBV01000001.1
9    RS_GCF_034087825.1~NZ_JAXDXQ010000151.1
Name: gtdb_ref, dtype: object
RS_GCF_002274615.1~NZ_LLAG01000108.1
'RS_GCF_002274615.1~NZ_LLAG01000108.1'


,asv_id,gtdb_ref,pct_identity,species,genus,kingdom,phylum,class,order,family,genus,species
0,PRJNA266806_ASV1,RS_GCF_002274615.1~NZ_LLAG01000108.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PRJNA266806_ASV8,RS_GCF_025187565.1~NZ_RIJA01000015.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PRJNA266806_ASV9,RS_GCF_934855955.1~NZ_CAKWVU010000076.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PRJNA266806_ASV2,RS_GCF_959025655.1~NZ_CAUFJN010000109.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PRJNA266806_ASV5,RS_GCF_002274615.1~NZ_LLAG01000108.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
print(hits["gtdb_ref"].isna().sum())
print(len(hits))

0
68341


In [18]:
with open("closed_ref_map.uc") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i > 5:
            break

'H\t234155\t150\t100.0\t-\t0\t0\t123D27M990I\tPRJNA266806_ASV1\tRS_GCF_002274615.1~NZ_LLAG01000108.1\n'
'H\t567544\t150\t100.0\t+\t0\t0\t23I150M34I\tPRJNA266806_ASV8\tRS_GCF_025187565.1~NZ_RIJA01000015.1\n'
'H\t634377\t150\t100.0\t+\t0\t0\t23I150M76I\tPRJNA266806_ASV9\tRS_GCF_934855955.1~NZ_CAKWVU010000076.1\n'
'H\t762279\t150\t100.0\t+\t0\t0\t25I150M39I\tPRJNA266806_ASV2\tRS_GCF_959025655.1~NZ_CAUFJN010000109.1\n'
'H\t234155\t150\t100.0\t-\t0\t0\t123D27M990I\tPRJNA266806_ASV5\tRS_GCF_002274615.1~NZ_LLAG01000108.1\n'
'H\t762267\t150\t100.0\t+\t0\t0\t25I150M27I\tPRJNA266806_ASV7\tGB_GCA_023447495.1~JAEXNC010000228.1\n'
'H\t660671\t150\t100.0\t+\t0\t0\t16I150M42I\tPRJNA266806_ASV3\tGB_GCA_022781305.1~JALETV010000081.1\n'


In [19]:

accession_to_tax = {}
with open("../curated_GTDB/ssu_all_r226.fna") as f:
    for line in f:
        if line.startswith(">"):
            header = line[1:].strip()
            accession = header.split(" ")[0]  # everything before first space
            accession_to_tax[accession] = header

tax_lookup_df = pd.DataFrame(
    [{"target": k, "full_header": v} for k, v in accession_to_tax.items()]
)

# Merge onto your existing hits table using the truncated accession as key
hits = hits.merge(tax_lookup_df, left_on="gtdb_ref", right_on="target", how="left")

# Now extract taxonomy from full_header instead of gtdb_ref
tax_pattern = (
    r"d__(?P<kingdom>[^;]+);"
    r"p__(?P<phylum>[^;]+);"
    r"c__(?P<class>[^;]+);"
    r"o__(?P<order>[^;]+);"
    r"f__(?P<family>[^;]+);"
    r"g__(?P<genus>[^;]+);"
    r"s__(?P<species>[^;\[]+)"
)

tax_df = hits["full_header"].str.extract(tax_pattern)
tax_df["species"] = tax_df["species"].str.strip()

hits = pd.concat([hits, tax_df], axis=1)
hits.to_csv("asv_to_gtdb_mapping.csv", index=False)
hits.head()

,asv_id,gtdb_ref,pct_identity,species,genus,kingdom,phylum,class,order,family,...,species,target,full_header,kingdom,phylum,class,order,family,genus,species
0,PRJNA266806_ASV1,RS_GCF_002274615.1~NZ_LLAG01000108.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,RS_GCF_002274615.1~NZ_LLAG01000108.1,RS_GCF_002274615.1~NZ_LLAG01000108.1 d__Bacter...,Bacteria,Bacillota,Bacilli,Staphylococcales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus
1,PRJNA266806_ASV8,RS_GCF_025187565.1~NZ_RIJA01000015.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,RS_GCF_025187565.1~NZ_RIJA01000015.1,RS_GCF_025187565.1~NZ_RIJA01000015.1 d__Bacter...,Bacteria,Bacillota,Bacilli,Lactobacillales,Streptococcaceae,Streptococcus,Streptococcus thermophilus
2,PRJNA266806_ASV9,RS_GCF_934855955.1~NZ_CAKWVU010000076.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,RS_GCF_934855955.1~NZ_CAKWVU010000076.1,RS_GCF_934855955.1~NZ_CAKWVU010000076.1 d__Bac...,Bacteria,Bacteroidota,Bacteroidia,Bacteroidales,Rikenellaceae,Alistipes,Alistipes putredinis
3,PRJNA266806_ASV2,RS_GCF_959025655.1~NZ_CAUFJN010000109.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,RS_GCF_959025655.1~NZ_CAUFJN010000109.1,RS_GCF_959025655.1~NZ_CAUFJN010000109.1 d__Bac...,Bacteria,Bacillota,Clostridia,Peptostreptococcales,Peptostreptococcaceae,Romboutsia,Romboutsia timonensis
4,PRJNA266806_ASV5,RS_GCF_002274615.1~NZ_LLAG01000108.1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,RS_GCF_002274615.1~NZ_LLAG01000108.1,RS_GCF_002274615.1~NZ_LLAG01000108.1 d__Bacter...,Bacteria,Bacillota,Bacilli,Staphylococcales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus
